# Notebook 06: Model Comparison

**Purpose**: Compare all 5 models using REAL results from training notebooks 01-05.

This notebook does NOT train any models.

In [1]:
import json, os
import numpy as np

RESULTS_DIR = r"D:\Nguyen-Anh-Viet\DeepLearning\DeepLearning\results"

# Load all results
model_names = ["baseline", "eca", "mish", "eca_mish", "rmsa"]
display_names = ["YOLO11s-P2", "+ ECA", "+ Mish", "+ ECA+Mish", "+ RMSA"]

results = {}
for name in model_names:
    path = os.path.join(RESULTS_DIR, f"{name}_metrics.json")
    if os.path.exists(path):
        with open(path) as f:
            results[name] = json.load(f)
        print(f"Loaded {name}: mAP50-95={results[name]['mAP50-95']:.4f}")
    else:
        print(f"WARNING: {path} not found!")

assert len(results) == 5, f"Expected 5 results, got {len(results)}"
print(f"\nAll {len(results)} result files loaded successfully!")


Loaded baseline: mAP50-95=0.1693
Loaded eca: mAP50-95=0.1638
Loaded mish: mAP50-95=0.1698
Loaded eca_mish: mAP50-95=0.1579
Loaded rmsa: mAP50-95=0.1546

All 5 result files loaded successfully!


## Main Comparison Table

In [2]:
# Create comparison table
print("=" * 120)
print(f"{'Model':<18} {'Params':>10} {'GFLOPs':>8} {'Size MB':>8} {'Precision':>10} {'Recall':>8} {'mAP50':>8} {'mAP50-95':>10} {'Lat ms':>8} {'FPS':>8}")
print("=" * 120)

for name, dname in zip(model_names, display_names):
    if name not in results:
        continue
    r = results[name]
    print(f"{dname:<18} {r['n_params']:>10,} {r['gflops']:>8.1f} {r['model_size_mb']:>8.1f} "
          f"{r['precision']:>10.4f} {r['recall']:>8.4f} {r['mAP50']:>8.4f} {r['mAP50-95']:>10.4f} "
          f"{r['latency_ms']:>8.1f} {r['fps']:>8.1f}")
print("=" * 120)


Model                  Params   GFLOPs  Size MB  Precision   Recall    mAP50   mAP50-95   Lat ms      FPS
YOLO11s-P2          9,575,292     29.0     18.7     0.4491   0.4095   0.3923     0.1693     34.4     29.0
+ ECA               9,575,292     29.0     18.7     0.4420   0.4259   0.3851     0.1638     22.3     44.9
+ Mish              9,575,292     29.0     18.7     0.4489   0.4143   0.3880     0.1698     21.4     46.7
+ ECA+Mish          9,575,292     29.0     18.7     0.4154   0.4290   0.3668     0.1579     21.8     46.0
+ RMSA              9,575,292     29.0     18.7     0.4107   0.4160   0.3681     0.1546     22.2     45.0


## Ablation Table

In [3]:
# Ablation table
baseline = results["baseline"]
base_map = baseline["mAP50-95"]
base_params = baseline["n_params"]
base_lat = baseline["latency_ms"]

print("=" * 130)
print(f"{'Model':<18} {'ECA':>5} {'Mish':>5} {'RMSA':>5} {'mAP50-95':>10} {'Delta mAP':>10} {'Params':>10} {'Delta Params':>13} {'Lat ms':>8} {'Delta Lat':>10}")
print("=" * 130)

ablation_config = {
    "baseline": {"eca": "No", "mish": "No", "rmsa": "No"},
    "eca": {"eca": "Yes", "mish": "No", "rmsa": "No"},
    "mish": {"eca": "No", "mish": "Yes", "rmsa": "No"},
    "eca_mish": {"eca": "Yes", "mish": "Yes", "rmsa": "No"},
    "rmsa": {"eca": "No", "mish": "No", "rmsa": "Yes"},
}

for name, dname in zip(model_names, display_names):
    if name not in results:
        continue
    r = results[name]
    cfg = ablation_config[name]
    d_map = r["mAP50-95"] - base_map
    d_params = r["n_params"] - base_params
    d_lat = r["latency_ms"] - base_lat
    
    print(f"{dname:<18} {cfg['eca']:>5} {cfg['mish']:>5} {cfg['rmsa']:>5} "
          f"{r['mAP50-95']:>10.4f} {d_map:>+10.4f} {r['n_params']:>10,} {d_params:>+13,} "
          f"{r['latency_ms']:>8.1f} {d_lat:>+10.1f}")
print("=" * 130)


Model                ECA  Mish  RMSA   mAP50-95  Delta mAP     Params  Delta Params   Lat ms  Delta Lat
YOLO11s-P2            No    No    No     0.1693    +0.0000  9,575,292            +0     34.4       +0.0
+ ECA                Yes    No    No     0.1638    -0.0055  9,575,292            +0     22.3      -12.2
+ Mish                No   Yes    No     0.1698    +0.0005  9,575,292            +0     21.4      -13.0
+ ECA+Mish           Yes   Yes    No     0.1579    -0.0114  9,575,292            +0     21.8      -12.7
+ RMSA                No    No   Yes     0.1546    -0.0148  9,575,292            +0     22.2      -12.2


## Comparison Plots

In [4]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Data
names_short = display_names[:len(results)]
maps = [results[n]["mAP50-95"] for n in model_names if n in results]
lats = [results[n]["latency_ms"] for n in model_names if n in results]
sizes = [results[n]["model_size_mb"] for n in model_names if n in results]
params = [results[n]["n_params"] / 1e6 for n in model_names if n in results]
map50s = [results[n]["mAP50"] for n in model_names if n in results]
precs = [results[n]["precision"] for n in model_names if n in results]

colors = ["#2196F3", "#4CAF50", "#FF9800", "#9C27B0", "#F44336"]

# Plot 1: mAP50-95
axes[0, 0].bar(names_short, maps, color=colors[:len(names_short)])
axes[0, 0].set_title("mAP50-95 by Model")
axes[0, 0].set_ylabel("mAP50-95")

# Plot 2: Latency
axes[0, 1].bar(names_short, lats, color=colors[:len(names_short)])
axes[0, 1].set_title("Latency by Model")
axes[0, 1].set_ylabel("Latency (ms)")

# Plot 3: Model Size
axes[0, 2].bar(names_short, sizes, color=colors[:len(names_short)])
axes[0, 2].set_title("Model Size by Model")
axes[0, 2].set_ylabel("Size (MB)")

# Plot 4: Parameters
axes[1, 0].bar(names_short, params, color=colors[:len(names_short)])
axes[1, 0].set_title("Parameters by Model")
axes[1, 0].set_ylabel("Parameters (M)")

# Plot 5: mAP50-95 vs Latency scatter
axes[1, 1].scatter(lats, maps, c=colors[:len(names_short)], s=100, zorder=5)
for i, name in enumerate(names_short):
    axes[1, 1].annotate(name, (lats[i], maps[i]), fontsize=8, ha='center', va='bottom')
axes[1, 1].set_xlabel("Latency (ms)")
axes[1, 1].set_ylabel("mAP50-95")
axes[1, 1].set_title("mAP50-95 vs Latency")

# Plot 6: mAP50-95 vs Model Size scatter
axes[1, 2].scatter(sizes, maps, c=colors[:len(names_short)], s=100, zorder=5)
for i, name in enumerate(names_short):
    axes[1, 2].annotate(name, (sizes[i], maps[i]), fontsize=8, ha='center', va='bottom')
axes[1, 2].set_xlabel("Model Size (MB)")
axes[1, 2].set_ylabel("mAP50-95")
axes[1, 2].set_title("mAP50-95 vs Model Size")

plt.tight_layout()
plot_path = os.path.join(RESULTS_DIR, "comparison_plots.png")
plt.savefig(plot_path, dpi=150, bbox_inches="tight")
print(f"Plots saved to {plot_path}")
plt.close()


Plots saved to D:\Nguyen-Anh-Viet\DeepLearning\DeepLearning\results\comparison_plots.png


## Per-Class Analysis

In [5]:
# Per-class AP comparison
class_names = ["Pothole", "Crack", "Manhole"]
has_per_class = all("per_class" in results[n] and results[n]["per_class"] for n in model_names if n in results)

if has_per_class:
    print("Per-Class AP50 Comparison:")
    print("=" * 90)
    print(f"{'Class':<12}", end="")
    for dname in display_names:
        print(f"{dname:>14}", end="")
    print()
    print("=" * 90)
    
    for cls in class_names:
        print(f"{cls:<12}", end="")
        for name in model_names:
            if name in results and cls in results[name].get("per_class", {}):
                print(f"{results[name]['per_class'][cls]['AP50']:>14.4f}", end="")
            else:
                print(f"{'N/A':>14}", end="")
        print()
    
    print("\nPer-Class AP50-95 Comparison:")
    print("=" * 90)
    print(f"{'Class':<12}", end="")
    for dname in display_names:
        print(f"{dname:>14}", end="")
    print()
    print("=" * 90)
    
    for cls in class_names:
        print(f"{cls:<12}", end="")
        for name in model_names:
            if name in results and cls in results[name].get("per_class", {}):
                print(f"{results[name]['per_class'][cls]['AP50-95']:>14.4f}", end="")
            else:
                print(f"{'N/A':>14}", end="")
        print()
else:
    print("Per-class metrics not available for all models")


Per-Class AP50 Comparison:
Class           YOLO11s-P2         + ECA        + Mish    + ECA+Mish        + RMSA
Pothole             0.3499        0.3468        0.3404        0.3251        0.3573
Crack               0.1949        0.2008        0.1948        0.1641        0.1855
Manhole             0.6321        0.6076        0.6289        0.6110        0.5616

Per-Class AP50-95 Comparison:
Class           YOLO11s-P2         + ECA        + Mish    + ECA+Mish        + RMSA
Pothole             0.1356        0.1302        0.1353        0.1278        0.1268
Crack               0.0682        0.0730        0.0678        0.0550        0.0645
Manhole             0.3043        0.2883        0.3064        0.2910        0.2724


## Experimental Questions - Evidence-Based Answers

In [6]:
print("=" * 80)
print("EXPERIMENTAL FINDINGS")
print("=" * 80)

baseline_r = results["baseline"]

# Answer each question
questions = [
    ("Does ECA improve YOLO11s-P2?", 
     "eca", lambda r: r["mAP50-95"] > baseline_r["mAP50-95"]),
    ("Does Mish improve YOLO11s-P2?",
     "mish", lambda r: r["mAP50-95"] > baseline_r["mAP50-95"]),
    ("Does ECA+Mish outperform ECA alone?",
     "eca_mish", lambda r: r["mAP50-95"] > results["eca"]["mAP50-95"]),
    ("Does ECA+Mish outperform Mish alone?",
     "eca_mish", lambda r: r["mAP50-95"] > results["mish"]["mAP50-95"]),
    ("Does RMSA outperform baseline?",
     "rmsa", lambda r: r["mAP50-95"] > baseline_r["mAP50-95"]),
]

for q, name, check in questions:
    if name in results:
        r = results[name]
        answer = "YES" if check(r) else "NO"
        delta = r["mAP50-95"] - baseline_r["mAP50-95"]
        print(f"\nQ: {q}")
        print(f"A: {answer} (mAP50-95: {r['mAP50-95']:.4f}, delta vs baseline: {delta:+.4f})")

# Best models
print("\n" + "=" * 80)
best_acc = max(model_names, key=lambda n: results[n]["mAP50-95"] if n in results else -1)
best_lat = min(model_names, key=lambda n: results[n]["latency_ms"] if n in results else float("inf"))
best_size = min(model_names, key=lambda n: results[n]["model_size_mb"] if n in results else float("inf"))

# Best tradeoff: highest mAP50-95 / latency ratio
best_tradeoff = max(model_names, key=lambda n: results[n]["mAP50-95"] / results[n]["latency_ms"] if n in results else -1)

print(f"\nBest mAP50-95: {display_names[model_names.index(best_acc)]} = {results[best_acc]['mAP50-95']:.4f}")
print(f"Fastest model: {display_names[model_names.index(best_lat)]} = {results[best_lat]['latency_ms']:.1f} ms")
print(f"Smallest model: {display_names[model_names.index(best_size)]} = {results[best_size]['model_size_mb']:.1f} MB")
print(f"Best accuracy-efficiency tradeoff: {display_names[model_names.index(best_tradeoff)]}")

print(f"\nBaseline mAP50-95: {baseline_r['mAP50-95']:.4f}")
print(f"Best custom mAP50-95: {results[best_acc]['mAP50-95']:.4f}")
print(f"Improvement: {results[best_acc]['mAP50-95'] - baseline_r['mAP50-95']:+.4f}")

# Discuss tradeoffs
print("\n" + "=" * 80)
print("TRADEOFF ANALYSIS")
print("=" * 80)
for name, dname in zip(model_names[1:], display_names[1:]):  # Skip baseline
    if name not in results:
        continue
    r = results[name]
    d_map = r["mAP50-95"] - baseline_r["mAP50-95"]
    d_lat = r["latency_ms"] - baseline_r["latency_ms"]
    d_lat_pct = (d_lat / baseline_r["latency_ms"]) * 100
    d_params = r["n_params"] - baseline_r["n_params"]
    d_params_pct = (d_params / baseline_r["n_params"]) * 100
    
    print(f"\n{dname}:")
    print(f"  mAP50-95 change: {d_map:+.4f}")
    print(f"  Latency change: {d_lat:+.1f} ms ({d_lat_pct:+.1f}%)")
    print(f"  Parameter change: {d_params:+,} ({d_params_pct:+.1f}%)")
    
    if d_map > 0 and d_lat_pct < 10:
        print(f"  -> Good tradeoff: accuracy gain with minimal latency cost")
    elif d_map > 0 and d_lat_pct >= 10:
        print(f"  -> Mixed: accuracy gain but significant latency increase")
    elif d_map <= 0:
        print(f"  -> Not justified: no accuracy improvement")


EXPERIMENTAL FINDINGS

Q: Does ECA improve YOLO11s-P2?
A: NO (mAP50-95: 0.1638, delta vs baseline: -0.0055)

Q: Does Mish improve YOLO11s-P2?
A: YES (mAP50-95: 0.1698, delta vs baseline: +0.0005)

Q: Does ECA+Mish outperform ECA alone?
A: NO (mAP50-95: 0.1579, delta vs baseline: -0.0114)

Q: Does ECA+Mish outperform Mish alone?
A: NO (mAP50-95: 0.1579, delta vs baseline: -0.0114)

Q: Does RMSA outperform baseline?
A: NO (mAP50-95: 0.1546, delta vs baseline: -0.0148)


Best mAP50-95: + Mish = 0.1698
Fastest model: + Mish = 21.4 ms
Smallest model: + Mish = 18.7 MB
Best accuracy-efficiency tradeoff: + Mish

Baseline mAP50-95: 0.1693
Best custom mAP50-95: 0.1698
Improvement: +0.0005

TRADEOFF ANALYSIS

+ ECA:
  mAP50-95 change: -0.0055
  Latency change: -12.2 ms (-35.3%)
  Parameter change: +0 (+0.0%)
  -> Not justified: no accuracy improvement

+ Mish:
  mAP50-95 change: +0.0005
  Latency change: -13.0 ms (-37.9%)
  Parameter change: +0 (+0.0%)
  -> Good tradeoff: accuracy gain with minim

## Final Conclusion

In [7]:
print("=" * 80)
print("FINAL CONCLUSION")  
print("=" * 80)
print()
print("This experiment compared 5 architectural variants of YOLO11s-P2 for")
print("road damage detection (potholes, cracks, manholes).")
print()

# Determine the best model
best_name = max(model_names, key=lambda n: results[n]["mAP50-95"])
best_dname = display_names[model_names.index(best_name)]
best_map = results[best_name]["mAP50-95"]
baseline_map = results["baseline"]["mAP50-95"]

print(f"Best Accuracy Model: {best_dname}")
print(f"  mAP50-95: {best_map:.4f} (baseline: {baseline_map:.4f}, delta: {best_map - baseline_map:+.4f})")
print()

fastest_name = min(model_names, key=lambda n: results[n]["latency_ms"])
fastest_dname = display_names[model_names.index(fastest_name)]
print(f"Fastest Model: {fastest_dname}")
print(f"  Latency: {results[fastest_name]['latency_ms']:.1f} ms ({results[fastest_name]['fps']:.1f} FPS)")
print()

print(f"Best Practical Tradeoff: {display_names[model_names.index(best_tradeoff)]}")
print()

if best_name == "baseline":
    print("FINDING: No custom architecture outperformed the baseline.")
    print("The standard YOLO11s-P2 architecture is already well-suited for this task.")
else:
    print(f"FINDING: {best_dname} achieved the highest mAP50-95.")
    if best_name == "rmsa":
        print(f"Proposed architecture: RD-YOLO11s-P2-RMSA")
    elif best_name == "eca_mish":
        print(f"Proposed architecture: RD-YOLO11s-P2-ECA-Mish")
    elif best_name == "eca":
        print(f"Proposed architecture: RD-YOLO11s-P2-ECA")
    elif best_name == "mish":
        print(f"Proposed architecture: RD-YOLO11s-P2-Mish")


FINAL CONCLUSION

This experiment compared 5 architectural variants of YOLO11s-P2 for
road damage detection (potholes, cracks, manholes).

Best Accuracy Model: + Mish
  mAP50-95: 0.1698 (baseline: 0.1693, delta: +0.0005)

Fastest Model: + Mish
  Latency: 21.4 ms (46.7 FPS)

Best Practical Tradeoff: + Mish

FINDING: + Mish achieved the highest mAP50-95.
Proposed architecture: RD-YOLO11s-P2-Mish
